# Stochastic mean-reverting drift

Momentum-like two-factor model, built by composition: log price follows drift plus noise, `dlog p = mu dt + sigma dW`, and the drift `mu` itself follows an OU process. The true drift is returned as hidden state `drift.mu`.

Fantasma simulates **log returns** (log price differences): one row per step, labelled at the end of the step. `fm.to_prices`, `fm.to_log_prices` and `fm.to_returns` turn them into other quantities.

Edit the parameters cell and re-run all: the same parameters and seed always give the same paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fantasma as fm

## Parameters

Model parameters are annualised; `dt` is the step length over a 365.25-day year.

In [ ]:
SEED = 42
N_PATHS = 5
START = "2024-01-01"
FREQ = "1d"  # Almanac frequency ("1m", "10m", "1h", "1d") or any pandas offset
N_STEPS = 365 * 5

SIGMA = 0.1  # price volatility
DRIFT_MEAN = 0.05  # long-run drift
DRIFT_HALF_LIFE = 2.0  # years
DRIFT_SIGMA = 0.3  # volatility of the drift
INITIAL_PRICE = 100.0
INITIAL_DRIFT = None  # None starts at DRIFT_MEAN
ASSET = "SD"

## Simulate

In [ ]:
model = fm.StochasticDrift(
    sigma=SIGMA,
    drift_mean=DRIFT_MEAN,
    drift_half_life=DRIFT_HALF_LIFE,
    drift_sigma=DRIFT_SIGMA,
    asset=ASSET,
    initial_price=INITIAL_PRICE,
    initial_drift=INITIAL_DRIFT,
)
model.links

In [ ]:
paths = fm.simulate(
    model,
    start=START,
    freq=FREQ,
    n_steps=N_STEPS,
    seed=SEED,
    n_paths=N_PATHS,
    return_hidden=True,
)
paths = paths if N_PATHS > 1 else [paths]  # one path is returned itself, not in a list
log_returns, hidden = paths[0]  # log returns; hidden state on the full grid
log_returns.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for i, (path_log_returns, _) in enumerate(paths):
    path_prices = fm.to_prices(path_log_returns)  # start included, from the recorded start price
    for asset in path_prices.columns:
        ax.plot(path_prices.index, path_prices[asset], lw=0.8, label=f"path {i} {asset}")
ax.set_title("Stochastic mean-reverting drift")
ax.set_ylabel("price")
if len(paths) * paths[0][0].shape[1] <= 10:
    ax.legend(fontsize=8)
plt.show()

## Hidden drift

In [ ]:
ax = plt.figure(figsize=(10, 2.5)).gca()
for i, (_, h) in enumerate(paths):
    ax.plot(h.index, h["drift.mu"], lw=0.8, label=f"path {i}")
ax.axhline(DRIFT_MEAN, color="k", lw=1, ls="--")
ax.set_title("true drift (annualised)")
plt.show()

## The same model, wired by hand

`StochasticDrift` is a `Composite`; building it from its parts gives identical log returns.

In [ ]:
drift = fm.OrnsteinUhlenbeck(DRIFT_MEAN, DRIFT_HALF_LIFE, DRIFT_SIGMA, x0=INITIAL_DRIFT, state_name="mu")
price = fm.Brownian(SIGMA, mu=DRIFT_MEAN, asset=ASSET, initial_price=INITIAL_PRICE)
wired = fm.Composite({"drift": drift, "price": price}, links={"price.mu": "drift.mu"})
by_hand = fm.simulate(wired, start=START, freq=FREQ, n_steps=N_STEPS, seed=SEED)
assert by_hand.equals(log_returns)
print("identical to path 0:", by_hand.equals(log_returns))